## Objective & Tasks:

Data Processing: Clean and integrate these datasets. This should include, but not be limited to,
handling missing values, duplicates, and possible outliers.


**1. Set up Delta auto-merge:**
Enable schema evolution before any write. This ensures new columns can be added safely when writing Silver.

**2. Read Bronze data**

**3. Trim whitespace & normalize strings.**

**4. Check for NULLS:** Replace all null values with type-appropriate safe defaults especially since no access to Data Source Owner.
No columns should be dropped.

- Strings → "UNKNOWN"
- Integers → -1
- Floats/Decimals → -1.0
- Boolean → False
- Others → None

**5. Check for duplicates** - use "CMPLID" key  -Defined as NHTSA’s internal unique sequence number. Intended to uniquely identify each complaint record: https://static.nhtsa.gov/odi/ffdd/cmpl/CMPL.txt.

**6.Repartition to spread data evenly for writing**
- To avoids skew or one huge file: split Delta files are roughly equal-sized, making writing, reading, and later optimizations faster.

**7. Write to Delta with mergeSchema for schema enforcement and evolution**
- Writes the Bronze DataFrame to a Delta path with schema evolution
- Registers it as a SQL-accessible table in the metastore

In [0]:
silver_df = spark.read.table("hive_metastore.nhtsa_complaints.bronze")
silver_df.printSchema()



## 1. Enable Delta Lake schema evolution for auto-merge (schema evolution)

In [0]:
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

## 2. Read Bronze data from Delta table

In [0]:
silver_df = spark.read.table("hive_metastore.nhtsa_complaints.bronze")

## 3. Trim whitespace

In [0]:
from pyspark.sql.functions import col, trim, lower

string_cols = [f.name for f in silver_df.schema.fields if f.dataType.simpleString() == 'string']
for c in string_cols:
    silver_df = silver_df.withColumn(c,(trim(col(c))))

## 4. Check for NULLS and handle them with apprropiate values

- Strings → "UNKNOWN"
- Integers → -1
- Floats/Decimals → -1.0
- Boolean → False
- Others → None

In [0]:
from pyspark.sql.functions import lit, when

# Define safe defaults for each column type
def get_safe_default(dtype):
    if dtype == "string":
        return "UNKNOWN"
    elif dtype in ["int", "bigint", "long", "short"]:
        return -1
    elif dtype in ["float", "double", "decimal"]:
        return -1.0
    elif dtype == "boolean":
        return False
    else:
        return None

# Fill all nulls with safe defaults and add null flag columns
from pyspark.sql.functions import when, col, lit

for field in silver_df.schema.fields:
    col_name = field.name
    dtype = field.dataType.simpleString()
    safe_default = get_safe_default(dtype)
    if safe_default is not None:
        silver_df = silver_df.withColumn(
            col_name,
            when(col(col_name).isNull(), lit(safe_default)).otherwise(col(col_name))
        )


display(silver_df)

In [0]:
## Data quality check for NULLS:

total_nulls = null_counts.select([sum(col(c)) for c in null_counts.columns]).collect()[0][0]

if total_nulls > 0:
    print("Nulls found")
else:
    print("No nulls found")


## 5. Check for duplicates 
- Use "CMPLID" key -Defined as NHTSA’s internal unique sequence number.

In [0]:
from pyspark.sql.functions import col, count

duplicates_df = silver_df.groupBy("CMPLID").count().filter(col("count") > 1)
display(duplicates_df)

## 6. Repartition: Spread data evenly for writing

In [0]:
#Repartition: Spread data evenly for writing → avoids skew or one huge file.
num_cores = spark.sparkContext.defaultParallelism
import builtins; target_partitions = builtins.max(32, num_cores * 8)
silver_df = silver_df.repartition(target_partitions)

### ## 7.Write to Delta with mergeSchema for schema enforcement and evolution

In [0]:
delta_path = "/mnt/delta/silver_nhtsa"
silver_table = "hive_metastore.nhtsa_complaints.silver"

# Write Delta with schema evolution
silver_df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .save(delta_path)

# Create table if not exists
spark.sql(f"CREATE TABLE IF NOT EXISTS {silver_table} USING DELTA LOCATION '{delta_path}'")

#Verify the table
display(spark.sql(f"SELECT * FROM {silver_table} LIMIT 10"))
